# Breast Cancer Detection System: Exploratory Data Analysis (EDA)
## Wisconsin Diagnostic Breast Cancer (WDBC) Dataset
**Author:** MCA AI & ML Research Team  
**Objective:** Systematic exploratory analysis of 569 cytological fine needle aspirate (FNA) samples across 30 nuclear morphological features.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

### 1. Load Dataset and Feature Inspection
In Scikit-learn's `load_breast_cancer`, target 0 corresponds to malignant and 1 to benign. For clinical interpretability and oncology metrics, we map `1 = Malignant` (positive class for sensitivity/recall) and `0 = Benign`.

In [2]:
raw_data = load_breast_cancer()
df = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)

# Target mapping: 1 for Malignant, 0 for Benign
df["diagnosis"] = np.where(raw_data.target == 0, "Malignant", "Benign")
df["target"] = np.where(raw_data.target == 0, 1, 0)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 2. Data Types, Missing Values, and Duplicates

In [3]:
print("Missing values per column:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nData Types Summary:")
print(df.dtypes.value_counts())

### 3. Class Distribution Analysis
Examines the clinical balance between Benign (non-cancerous) and Malignant (cancerous) cases.

In [4]:
class_counts = df["diagnosis"].value_counts()
class_pct = df["diagnosis"].value_counts(normalize=True) * 100

print(pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pct.round(2)}))

plt.figure(figsize=(7, 4))
ax = sns.countplot(x="diagnosis", data=df, palette=["#3b82f6", "#ef4444"])
plt.title("Distribution of Diagnosis Classes (Benign vs Malignant)", fontsize=13, fontweight="bold")
for p in ax.patches:
    ax.annotate(f'{p.get_height()} ({p.get_height()/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.show()

### 4. Descriptive Statistics
Key summary statistics comparing mean measurements across classes.

In [5]:
mean_cols = [c for c in df.columns if "mean" in c]
display_stats = df.groupby("diagnosis")[mean_cols].mean().T
display_stats["Ratio (M/B)"] = (display_stats["Malignant"] / display_stats["Benign"]).round(2)
display_stats

### 5. Correlation Heatmap Analysis
Calculates Pearson correlation coefficients across the 10 nuclear mean characteristics.

In [6]:
plt.figure(figsize=(11, 9))
mean_corr = df[mean_cols].corr()
mask = np.triu(np.ones_like(mean_corr, dtype=bool))
sns.heatmap(mean_corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8})
plt.title("Inter-Feature Pearson Correlation (Mean Morphologies)", fontsize=13, fontweight="bold")
plt.show()

### 6. Feature Distributions: Histograms and Box Plots
Visualizing separation power of radius, concave points, and area between benign and malignant cases.

In [7]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
selected_features = ["mean radius", "mean concave points", "mean area", "worst radius", "worst concave points", "worst area"]

for i, feat in enumerate(selected_features):
    row, col = i // 3, i % 3
    sns.boxplot(x="diagnosis", y=feat, data=df, ax=axes[row, col], palette=["#3b82f6", "#ef4444"])
    axes[row, col].set_title(f"{feat.title()} by Diagnosis", fontweight="bold")

plt.tight_layout()
plt.show()

### 7. Major Clinical & Statistical Observations
1. **Nuclear Hypertrophy & Area Disparity:** Malignant nuclei exhibit an average area of 978.38 µm² compared to 462.79 µm² in benign nuclei (a 2.11× enlargement ratio), signifying uncontrolled cellular dyskaryosis.
2. **Irregular Perimeter & Concavity:** Mean concave points and concavity demonstrate sharp discriminative separation with minimal distribution overlap, serving as powerful diagnostic predictors.
3. **Collinearity Considerations:** `mean radius`, `mean perimeter`, and `mean area` have Pearson correlation coefficients > 0.98. Tree ensembles (Random Forest, Gradient Boosting) handle collinearity naturally, while regularized models (Logistic Regression L2, SVM RBF) benefit from StandardScaler feature normalization.
4. **Class Balance:** With 357 benign (62.7%) and 212 malignant (37.3%), stratified k-fold cross-validation is mandatory to prevent distribution drift between training and evaluation folds.